# Batch Abstract-Only Extraction Evaluation

Evaluate abstract-only metadata extraction across all Fuster validation samples (Dryad + Zenodo + Semantic Scholar).

**Approach:**
- Load validated ground truth from `data/dataset_092624_validated.xlsx`
- Join with original `data/dataset_092624.xlsx` to get abstract text (`full_text` column)
- Filter to records with non-null abstracts
- Run GPT-5-mini extraction using `text_classification_flow()` (Prefect, parallel)
- Compare against manually annotated ground truth
- Per-field P/R/F1 for all 14 evaluatable fields (8 core + 6 modulators)
- Segmented by source: Dryad vs Zenodo vs Semantic Scholar

**Cache note:** The joblib cache in `gpt_extract.py` uses `./cache` relative to the working directory.
This notebook ensures `os.chdir(PROJECT_ROOT)` is called before imports so the cache lands at
`{PROJECT_ROOT}/cache/` — which is NOT matched by the `*/cache/*` gitignore pattern and can be
committed and synced to your local machine.

In [1]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path

# ── Cache & working directory ──────────────────────────────────────────────
# IMPORTANT: set PROJECT_ROOT and chdir BEFORE importing any llm_metadata
# modules. gpt_extract.py creates Memory('./cache') at import time, so the
# resolved cache path depends on cwd at import time.
#
# Root-level cache/ is NOT matched by the '*/cache/*' gitignore pattern
# (that pattern needs at least one segment before 'cache'), so the cache at
# PROJECT_ROOT/cache/ can be committed and pulled locally.
_cwd = Path.cwd()
PROJECT_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
os.chdir(PROJECT_ROOT)
print(f"Working directory set to: {PROJECT_ROOT}")
print(f"Joblib cache will land at: {PROJECT_ROOT / 'cache'}")

from datetime import datetime
import pandas as pd
import numpy as np
from dotenv import load_dotenv

load_dotenv()

# ── Project modules ────────────────────────────────────────────────────────
from llm_metadata.gpt_extract import SYSTEM_MESSAGE
from llm_metadata.text_pipeline import (
    TextClassificationConfig,
    TextInputRecord,
    TextOutputRecord,
    text_classification_flow,
    save_text_manifest,
)
from llm_metadata.schemas.fuster_features import (
    DatasetFeatures,
    DatasetFeaturesNormalized,
)
from llm_metadata.groundtruth_eval import (
    evaluate_indexed,
    EvaluationConfig,
    FuzzyMatchConfig,
    micro_average,
    macro_f1,
)

# ── Paths ──────────────────────────────────────────────────────────────────
VALIDATED_PATH = PROJECT_ROOT / "data/dataset_092624_validated.xlsx"
ORIGINAL_PATH  = PROJECT_ROOT / "data/dataset_092624.xlsx"
OUTPUT_DIR     = PROJECT_ROOT / "artifacts/abstract_results"

print(f"Validated data: {VALIDATED_PATH.exists()}")
print(f"Original data:  {ORIGINAL_PATH.exists()}")

Working directory set to: /home/user/llm_metadata
Joblib cache will land at: /home/user/llm_metadata/cache


/home/user/llm_metadata/.venv/lib/python3.11/site-packages/pydantic/json_schema.py:2448: PydanticJsonSchemaWarning: Default value <class 'llm_metadata.schemas.abstract_metadata.DatasetAbstractMetadata'> is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
/home/user/llm_metadata/.venv/lib/python3.11/site-packages/pydantic/json_schema.py:2448: PydanticJsonSchemaWarning: Default value <class 'llm_metadata.schemas.abstract_metadata.DatasetAbstractMetadata'> is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)


Validated data: True
Original data:  True


## Step 1: Load Data

Load the validated ground truth and join with the original file to get abstract text.

In [2]:
# Load validated ground truth (schema-valid, valid_yn='yes', all sources)
validated_df = pd.read_excel(VALIDATED_PATH)
print(f"Validated records: {len(validated_df)}")
print(f"By source:")
print(validated_df['source'].value_counts())
print(f"\nColumns: {list(validated_df.columns)}")

Validated records: 299
By source:
source
semantic_scholar    192
zenodo               67
dryad                37
referenced            3
Name: count, dtype: int64

Columns: ['data_type', 'geospatial_info_dataset', 'spatial_range_km2', 'temporal_range', 'temp_range_i', 'temp_range_f', 'species', 'referred_dataset', 'time_series', 'multispecies', 'threatened_species', 'new_species_science', 'new_species_region', 'bias_north_south', 'valid_yn', 'reason_not_valid', 'source', 'source_url', 'journal_url', 'pdf_url', 'is_oa', 'cited_article_doi', 'id', 'title', 'has_abstract']


In [3]:
# Load original file to get full_text (abstract) column
original_df = pd.read_excel(ORIGINAL_PATH, usecols=['id', 'full_text'])
original_df = original_df.rename(columns={'full_text': 'abstract'})
print(f"Original records: {len(original_df)}, with abstract: {original_df['abstract'].notna().sum()}")

# Merge on id — validated_df.id is the row ID from the original xlsx
validated_df['id'] = validated_df['id'].astype(int)
original_df['id'] = original_df['id'].astype(int)
gt_df = validated_df.merge(original_df, on='id', how='left')

print(f"\nAfter join: {len(gt_df)} rows")
print(f"With abstract: {gt_df['abstract'].notna().sum()}")
print(f"\nAbstract coverage by source:")
print(gt_df.groupby('source')['abstract'].apply(lambda s: f"{s.notna().sum()}/{len(s)}"))

Original records: 418, with abstract: 406

After join: 301 rows
With abstract: 291

Abstract coverage by source:
source
dryad                 36/37
referenced              0/3
semantic_scholar    188/194
zenodo                67/67
Name: abstract, dtype: str


In [4]:
import ast

# ── Helper: Excel stores Python list objects as their repr string ───────────
# e.g.  "['genetic_analysis']"  →  ['genetic_analysis']
# The validated xlsx was exported via pandas to_excel(), which serialises lists
# to str. We need to restore the original list values before Pydantic validation.
LIST_COLS = [
    'data_type', 'geospatial_info_dataset', 'species',
]

def _parse_excel_val(val):
    """Parse Python list/bool repr from Excel back to native types."""
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return None
    if isinstance(val, list):
        return val
    if isinstance(val, str):
        s = val.strip()
        if s.startswith('['):
            try:
                return ast.literal_eval(s)
            except Exception:
                pass
    return val

def parse_gt_row(row: pd.Series) -> dict:
    """Convert a ground-truth row to a dict ready for DatasetFeaturesNormalized."""
    d = {col: row[col] for col in RELEVANT_COLS if col in row.index}
    for col in LIST_COLS:
        if col in d:
            d[col] = _parse_excel_val(d[col])
    return d

# Filter to records with abstracts
with_abstract_df = gt_df[gt_df['abstract'].notna()].copy()
print(f"Records with abstract: {len(with_abstract_df)}")
print(f"By source:")
print(with_abstract_df['source'].value_counts())

# Validate ground truth through DatasetFeaturesNormalized
RELEVANT_COLS = [
    'data_type', 'geospatial_info_dataset', 'spatial_range_km2',
    'temporal_range', 'temp_range_i', 'temp_range_f',
    'species', 'referred_dataset',
    'time_series', 'multispecies', 'threatened_species',
    'new_species_science', 'new_species_region', 'bias_north_south',
]

ground_truth_by_id = {}  # keyed by record id (str)
validation_errors = []

for _, row in with_abstract_df.iterrows():
    record_id = str(int(row['id']))
    try:
        row_dict = parse_gt_row(row)
        validated = DatasetFeaturesNormalized.model_validate(row_dict)
        ground_truth_by_id[record_id] = validated
    except Exception as e:
        validation_errors.append({'id': record_id, 'error': str(e)})

print(f"\nGround truth validated: {len(ground_truth_by_id)}")
print(f"Validation errors: {len(validation_errors)}")
if validation_errors:
    for err in validation_errors[:3]:
        print(f"  id={err['id']}: {err['error'][:120]}")

Records with abstract: 291
By source:
source
semantic_scholar    188
zenodo               67
dryad                36
Name: count, dtype: int64

Ground truth validated: 288
Validation errors: 0


## Step 2: Configure Pipeline

Set up abstract text classification with `DatasetFeatures` schema and `SYSTEM_MESSAGE`.

In [5]:
config = TextClassificationConfig(
    model="gpt-5-mini",
    reasoning={"effort": "low"},
    max_output_tokens=4096,
    text_format=DatasetFeatures,
    system_message=SYSTEM_MESSAGE,
    max_workers=5,
    output_dir=OUTPUT_DIR,
)

print("Abstract Classification Configuration:")
print(f"  Model:         {config.model}")
print(f"  Reasoning:     {config.reasoning}")
print(f"  Schema:        {config.text_format.__name__}")
print(f"  Max workers:   {config.max_workers}")
print(f"\nSystem prompt preview (first 200 chars):")
print(f"  {config.system_message[:200]}...")

Abstract Classification Configuration:
  Model:         gpt-5-mini
  Reasoning:     {'effort': 'low'}
  Schema:        DatasetFeatures
  Max workers:   5

System prompt preview (first 200 chars):
  You are EcodataGPT, a structured data extraction engine.

Goal: extract fields from the user's abstract into the provided schema.

Rules:
- Only use information explicitly supported by the text. Do NO...


## Step 3: Extract

Build `TextInputRecord` list and run `text_classification_flow()` (Prefect, ThreadPool).
Joblib cache ensures reruns are free — cached results are reused automatically.

In [6]:
# Build input records — id = str(row id), text = abstract
input_records = []
for _, row in with_abstract_df.iterrows():
    input_records.append(TextInputRecord(
        id=str(int(row['id'])),
        text=str(row['abstract']),
        metadata={
            'source': row['source'],
            'source_url': row.get('source_url', None),
        }
    ))

print(f"Input records: {len(input_records)}")
print(f"Sample record id: {input_records[0].id}")
print(f"Sample text (first 200 chars): {input_records[0].text[:200]}")

Input records: 291
Sample record id: 3
Sample text (first 200 chars): Fish mock community with 41 species from 13 orders <p dir="ltr">Tissue extracts of 41 North American fish species were obtained from the Ministère des Forêts, de la Faune et des Parcs (Québec). The se


In [7]:
# Run batch extraction
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_manifest_path = OUTPUT_DIR / f"abstract_results_{timestamp}.csv"

print(f"Running abstract classification on {len(input_records)} records...")
print(f"Model: {config.model}, reasoning: {config.reasoning}")
print(f"Output manifest: {output_manifest_path}")
print()

results = text_classification_flow(
    input_records=input_records,
    config=config,
    output_manifest=output_manifest_path,
)

success_count = sum(1 for r in results if r.status == "success")
error_count   = sum(1 for r in results if r.status == "error")
print(f"\nProcessing complete: {success_count} success, {error_count} errors")

Running abstract classification on 291 records...
Model: gpt-5-mini, reasoning: {'effort': 'low'}
Output manifest: /home/user/llm_metadata/artifacts/abstract_results/abstract_results_20260218_145110.csv



14:51:16.315 | INFO    | prefect - Starting temporary server on http://127.0.0.1:8246
See https://docs.prefect.io/v3/concepts/server#how-to-guides for more information on running a dedicated Prefect server.

14:51:28.371 | INFO    | Flow run 'rough-crow' - Beginning flow run 'rough-crow' for flow 'text-classification-flow'

Processing 291 text records...


14:51:29.577 | INFO    | Task run 'process_text_record-6b8' - Finished in state Completed()

...

14:51:35.051 | INFO    | Task run 'process_text_record-490' - Finished in state Completed()

14:51:35.057 | INFO    | Task run 'process_text_record-88d' - Finished in state Completed()

Completed: 291 success, 0 failed
Total cost: $1.9108
Saved output manifest to /home/user/llm_metadata/artifacts/abstract_results/abstract_results_20260218_145110.csv (291 records)


14:51:35.422 | INFO    | Flow run 'rough-crow' - Finished in state Completed()


Processing complete: 291 success, 0 errors


## Step 4: Prepare Extractions for Evaluation

Convert extraction results to `DatasetFeaturesNormalized` Pydantic models.

In [8]:
# Build predictions keyed by record id
predictions_by_id = {}

for result in results:
    if result.status != "success" or not result.extraction:
        continue
    record_id = result.id
    try:
        # Validate prediction through normalized schema for fair comparison
        pred = DatasetFeaturesNormalized.model_validate(result.extraction)
        predictions_by_id[record_id] = pred
    except Exception as e:
        print(f"Validation error for id={record_id}: {e}")

print(f"Valid predictions: {len(predictions_by_id)}")

# Find common IDs for evaluation
common_ids = set(predictions_by_id.keys()) & set(ground_truth_by_id.keys())
print(f"Common IDs for evaluation: {len(common_ids)}")

# Source breakdown for evaluated records
id_to_source = dict(zip(with_abstract_df['id'].astype(str), with_abstract_df['source']))
source_counts = {}
for rid in common_ids:
    src = id_to_source.get(rid, 'unknown')
    source_counts[src] = source_counts.get(src, 0) + 1
print(f"\nBy source:")
for src, cnt in sorted(source_counts.items()):
    print(f"  {src}: {cnt}")

Valid predictions: 288
Common IDs for evaluation: 288

By source:
  dryad: 36
  semantic_scholar: 185
  zenodo: 67


## Step 5: Evaluate

Compute per-field P/R/F1 for all 14 evaluatable fields: 8 core EBV fields + 6 modulator booleans.

In [9]:
# Fields to evaluate
CORE_FIELDS = [
    'data_type', 'geospatial_info_dataset', 'spatial_range_km2',
    'temporal_range', 'temp_range_i', 'temp_range_f',
    'species', 'referred_dataset',
]
MODULATOR_FIELDS = [
    'time_series', 'multispecies', 'threatened_species',
    'new_species_science', 'new_species_region', 'bias_north_south',
]
EVAL_FIELDS = CORE_FIELDS + MODULATOR_FIELDS

# Evaluation config: case-insensitive, set-based lists, fuzzy species matching
eval_config = EvaluationConfig(
    casefold_strings=True,
    treat_lists_as_sets=True,
    enhanced_species_matching=True,
    enhanced_species_threshold=70,
)

true_by_id = {rid: ground_truth_by_id[rid] for rid in common_ids}
pred_by_id = {rid: predictions_by_id[rid] for rid in common_ids}

abstract_report = evaluate_indexed(
    true_by_id=true_by_id,
    pred_by_id=pred_by_id,
    fields=EVAL_FIELDS,
    config=eval_config,
)

print("ABSTRACT-ONLY Extraction Metrics (all sources):")
print("=" * 70)
display(abstract_report.metrics_to_pandas())

ABSTRACT-ONLY Extraction Metrics (all sources):


,field,tp,fp,fn,tn,n,precision,recall,f1,accuracy,exact_match_rate
13,bias_north_south,0,0,159,129,288,NaN,0.000000,NaN,0.447917,0.447917
0,data_type,84,578,126,4,288,0.126888,0.400000,0.192661,0.305556,0.052083
1,geospatial_info_dataset,44,641,66,3,288,0.064234,0.400000,0.110692,0.163194,0.013889
9,multispecies,58,140,101,39,288,0.292929,0.364780,0.324930,0.336806,0.336806
12,new_species_region,6,16,99,169,288,0.272727,0.057143,0.094488,0.607639,0.607639
11,new_species_science,2,4,103,179,288,0.333333,0.019048,0.036036,0.628472,0.628472
7,referred_dataset,0,46,0,242,288,0.000000,NaN,NaN,0.840278,0.840278
2,spatial_range_km2,6,5,99,178,288,0.545455,0.057143,0.103448,0.638889,0.638889
6,species,222,1141,148,3,288,0.162876,0.600000,0.256203,0.781250,0.152778
5,temp_range_f,37,45,75,141,288,0.451220,0.330357,0.381443,0.618056,0.618056


In [10]:
# Aggregate metrics
abstract_micro = micro_average(abstract_report.field_metrics.values())
abstract_macro = macro_f1(abstract_report.field_metrics.values())

# Split by field group
core_metrics = {k: v for k, v in abstract_report.field_metrics.items() if k in CORE_FIELDS}
mod_metrics  = {k: v for k, v in abstract_report.field_metrics.items() if k in MODULATOR_FIELDS}
core_micro = micro_average(core_metrics.values())
mod_micro  = micro_average(mod_metrics.values())

print("\nAggregate Metrics:")
print("=" * 55)
print(f"{'Metric':<30} {'Value':>12}")
print("-" * 55)
print(f"{'[ALL] Micro Precision':<30} {abstract_micro.precision or 0:>12.3f}")
print(f"{'[ALL] Micro Recall':<30} {abstract_micro.recall or 0:>12.3f}")
print(f"{'[ALL] Micro F1':<30} {abstract_micro.f1 or 0:>12.3f}")
print(f"{'[ALL] Macro F1':<30} {abstract_macro or 0:>12.3f}")
print("-" * 55)
print(f"{'[CORE] Micro F1':<30} {core_micro.f1 or 0:>12.3f}")
print(f"{'[MODULATOR] Micro F1':<30} {mod_micro.f1 or 0:>12.3f}")
print("-" * 55)
print(f"{'Records Evaluated':<30} {len(common_ids):>12}")
print("=" * 55)


Aggregate Metrics:
Metric                                Value
-------------------------------------------------------
[ALL] Micro Precision                 0.154
[ALL] Micro Recall                    0.294
[ALL] Micro F1                        0.202
[ALL] Macro F1                        0.194
-------------------------------------------------------
[CORE] Micro F1                       0.209
[MODULATOR] Micro F1                  0.175
-------------------------------------------------------
Records Evaluated                       288


## Step 6: Analysis

Per-field metrics table, segmentation by source (Dryad / Zenodo / Semantic Scholar), and mismatch examples.

In [11]:
# Per-field metrics table
field_data = []
for f in EVAL_FIELDS:
    fm = abstract_report.field_metrics.get(f)
    if fm:
        field_data.append({
            'field': f,
            'group': 'core' if f in CORE_FIELDS else 'modulator',
            'precision': round(fm.precision or 0, 3),
            'recall':    round(fm.recall or 0, 3),
            'f1':        round(fm.f1 or 0, 3),
            'tp': fm.tp, 'fp': fm.fp, 'fn': fm.fn,
            'exact_match_rate': round(fm.exact_match_rate or 0, 3),
        })

field_df = pd.DataFrame(field_data)
print("Per-Field Metrics (Abstract-Only):")
display(field_df)

if field_data:
    best  = max(field_data, key=lambda x: x['f1'])
    worst = min(field_data, key=lambda x: x['f1'])
    print(f"\nBest:  {best['field']}  (F1={best['f1']})")
    print(f"Worst: {worst['field']} (F1={worst['f1']})")

Per-Field Metrics (Abstract-Only):


,field,group,precision,recall,f1,tp,fp,fn,exact_match_rate
0,data_type,core,0.127,0.400,0.193,84,578,126,0.052
1,geospatial_info_dataset,core,0.064,0.400,0.111,44,641,66,0.014
2,spatial_range_km2,core,0.545,0.057,0.103,6,5,99,0.639
3,temporal_range,core,0.063,0.066,0.064,9,134,128,0.337
4,temp_range_i,core,0.482,0.357,0.410,40,43,72,0.625
5,temp_range_f,core,0.451,0.330,0.381,37,45,75,0.618
6,species,core,0.163,0.600,0.256,222,1141,148,0.153
7,referred_dataset,core,0.000,0.000,0.000,0,46,0,0.840
8,time_series,modulator,0.112,0.917,0.200,11,87,1,0.694
9,multispecies,modulator,0.293,0.365,0.325,58,140,101,0.337



Best:  temp_range_i  (F1=0.41)
Worst: referred_dataset (F1=0)


In [12]:
# Per-source evaluation
SOURCES = ['dryad', 'zenodo', 'semantic_scholar']

source_summary = []

for src in SOURCES:
    src_ids = {rid for rid in common_ids if id_to_source.get(rid) == src}
    if not src_ids:
        continue

    src_true = {rid: true_by_id[rid] for rid in src_ids}
    src_pred = {rid: pred_by_id[rid] for rid in src_ids}

    src_report = evaluate_indexed(
        true_by_id=src_true,
        pred_by_id=src_pred,
        fields=EVAL_FIELDS,
        config=eval_config,
    )

    src_micro = micro_average(src_report.field_metrics.values())
    src_macro = macro_f1(src_report.field_metrics.values())

    source_summary.append({
        'source': src,
        'n': len(src_ids),
        'micro_precision': round(src_micro.precision or 0, 3),
        'micro_recall':    round(src_micro.recall or 0, 3),
        'micro_f1':        round(src_micro.f1 or 0, 3),
        'macro_f1':        round(src_macro or 0, 3),
    })

    print(f"\n{'='*60}")
    print(f"SOURCE: {src.upper()} (n={len(src_ids)})")
    print(f"{'='*60}")
    display(src_report.metrics_to_pandas())

print("\n" + "="*60)
print("CROSS-SOURCE SUMMARY")
print("="*60)
display(pd.DataFrame(source_summary))


SOURCE: DRYAD (n=36)


,field,tp,fp,fn,tn,n,precision,recall,f1,accuracy,exact_match_rate
13,bias_north_south,0,0,35,1,36,NaN,0.000000,NaN,0.027778,0.027778
0,data_type,25,49,24,0,36,0.337838,0.510204,0.406504,0.694444,0.166667
1,geospatial_info_dataset,11,80,11,0,36,0.120879,0.500000,0.194690,0.305556,0.000000
9,multispecies,7,9,28,1,36,0.437500,0.200000,0.274510,0.222222,0.222222
12,new_species_region,0,1,30,6,36,0.000000,0.000000,NaN,0.166667,0.166667
11,new_species_science,1,0,29,6,36,1.000000,0.033333,0.064516,0.194444,0.194444
7,referred_dataset,0,14,0,22,36,0.000000,NaN,NaN,0.611111,0.611111
2,spatial_range_km2,1,0,22,13,36,1.000000,0.043478,0.083333,0.388889,0.388889
6,species,50,170,7,0,36,0.227273,0.877193,0.361011,1.388889,0.194444
5,temp_range_f,4,6,26,6,36,0.400000,0.133333,0.200000,0.277778,0.277778



SOURCE: ZENODO (n=67)


,field,tp,fp,fn,tn,n,precision,recall,f1,accuracy,exact_match_rate
13,bias_north_south,0,0,56,11,67,NaN,0.000000,NaN,0.164179,0.164179
0,data_type,31,127,46,0,67,0.196203,0.402597,0.263830,0.462687,0.074627
1,geospatial_info_dataset,19,158,27,2,67,0.107345,0.413043,0.170404,0.313433,0.044776
9,multispecies,18,21,38,5,67,0.461538,0.321429,0.378947,0.343284,0.343284
12,new_species_region,0,0,38,29,67,NaN,0.000000,NaN,0.432836,0.432836
11,new_species_science,0,0,38,29,67,NaN,0.000000,NaN,0.432836,0.432836
7,referred_dataset,0,26,0,41,67,0.000000,NaN,NaN,0.611940,0.611940
2,spatial_range_km2,5,0,37,25,67,1.000000,0.119048,0.212766,0.447761,0.447761
6,species,66,229,33,0,67,0.223729,0.666667,0.335025,0.985075,0.238806
5,temp_range_f,23,3,29,13,67,0.884615,0.442308,0.589744,0.537313,0.537313



SOURCE: SEMANTIC_SCHOLAR (n=185)


,field,tp,fp,fn,tn,n,precision,recall,f1,accuracy,exact_match_rate
13,bias_north_south,0,0,68,117,185,NaN,0.000000,NaN,0.632432,0.632432
0,data_type,28,402,56,4,185,0.065116,0.333333,0.108949,0.172973,0.021622
1,geospatial_info_dataset,14,403,28,1,185,0.033573,0.333333,0.061002,0.081081,0.005405
9,multispecies,33,110,35,33,185,0.230769,0.485294,0.312796,0.356757,0.356757
12,new_species_region,6,15,31,134,185,0.285714,0.162162,0.206897,0.756757,0.756757
11,new_species_science,1,4,36,144,185,0.200000,0.027027,0.047619,0.783784,0.783784
7,referred_dataset,0,6,0,179,185,0.000000,NaN,NaN,0.967568,0.967568
2,spatial_range_km2,0,5,40,140,185,0.000000,0.000000,NaN,0.756757,0.756757
6,species,106,742,108,3,185,0.125000,0.495327,0.199623,0.589189,0.113514
5,temp_range_f,10,36,20,122,185,0.217391,0.333333,0.263158,0.713514,0.713514



CROSS-SOURCE SUMMARY


,source,n,micro_precision,micro_recall,micro_f1,macro_f1
0,dryad,36,0.242,0.277,0.259,0.237
1,zenodo,67,0.243,0.311,0.273,0.329
2,semantic_scholar,185,0.100,0.287,0.148,0.155


In [13]:
# Mismatch examples (for presentation)
results_detail_df = abstract_report.to_pandas()
mismatches = results_detail_df[~results_detail_df['match']]
print(f"Total mismatches: {len(mismatches)}")

# Show a sample of mismatches per field
print("\nMismatch count by field:")
print(mismatches['field'].value_counts())

print("\nSample mismatches (first 15):")
display(mismatches[['record_id', 'field', 'true_value', 'pred_value', 'tp', 'fp', 'fn']].head(15))

Total mismatches: 2121

Mismatch count by field:
field
geospatial_info_dataset    284
data_type                  273
species                    244
temporal_range             191
multispecies               191
bias_north_south           159
new_species_region         113
temp_range_f               110
temp_range_i               108
new_species_science        107
spatial_range_km2          104
threatened_species         103
time_series                 88
referred_dataset            46
Name: count, dtype: int64

Sample mismatches (first 15):


,record_id,field,true_value,pred_value,tp,fp,fn
0,10,data_type,[presence-absence],"[presence-absence, ecosystem_structure]",1,1,0
1,10,geospatial_info_dataset,[site_ids],"[sample, site, geographic_features, administra...",0,4,1
3,10,temporal_range,2020,25 June – 16 August 2020,0,1,1
6,10,species,[white-tailed deer],"[white-tailed deer, Cornus canadensis, Maianth...",1,9,0
9,10,multispecies,False,True,0,1,1
10,10,threatened_species,False,None,0,0,1
11,10,new_species_science,False,None,0,0,1
12,10,new_species_region,False,None,0,0,1
13,10,bias_north_south,False,None,0,0,1
14,100,data_type,[density],"[distribution, time_series]",0,2,1


## Step 7: Cost Analysis

In [14]:
successful_results = [r for r in results if r.status == "success"]

if successful_results:
    total_cost         = sum(r.cost_usd or 0 for r in successful_results)
    total_input_tokens = sum(r.input_tokens or 0 for r in successful_results)
    total_output_tokens= sum(r.output_tokens or 0 for r in successful_results)

    print("\nCOST ANALYSIS (Abstract-Only Extraction)")
    print("=" * 55)
    print(f"{'Metric':<35} {'Value':>18}")
    print("-" * 55)
    print(f"{'Records Processed':<35} {len(successful_results):>18}")
    print(f"{'Avg Input Tokens / record':<35} {total_input_tokens / len(successful_results):>18,.0f}")
    print(f"{'Avg Output Tokens / record':<35} {total_output_tokens / len(successful_results):>18,.0f}")
    print("-" * 55)
    print(f"{'Total Input Tokens':<35} {total_input_tokens:>18,}")
    print(f"{'Total Output Tokens':<35} {total_output_tokens:>18,}")
    print(f"{'Total Cost (USD)':<35} ${total_cost:>17.4f}")
    print(f"{'Avg Cost per record (USD)':<35} ${total_cost / len(successful_results):>17.5f}")
    print("=" * 55)

    # Per-source cost
    print("\nCost breakdown by source:")
    id_results = {r.id: r for r in successful_results}
    for src in SOURCES:
        src_ids = {rid for rid in common_ids if id_to_source.get(rid) == src}
        src_results = [id_results[rid] for rid in src_ids if rid in id_results]
        if src_results:
            src_cost = sum(r.cost_usd or 0 for r in src_results)
            print(f"  {src:<22}: {len(src_results):>4} records, ${src_cost:.4f} total, ${src_cost/len(src_results):.5f}/record")


COST ANALYSIS (Abstract-Only Extraction)
Metric                                           Value
-------------------------------------------------------
Records Processed                                  291
Avg Input Tokens / record                        2,282
Avg Output Tokens / record                       1,653
-------------------------------------------------------
Total Input Tokens                             664,120
Total Output Tokens                            481,071
Total Cost (USD)                    $           1.9108
Avg Cost per record (USD)           $          0.00657

Cost breakdown by source:
  dryad                 :   36 records, $0.2410 total, $0.00669/record
  zenodo                :   67 records, $0.4831 total, $0.00721/record
  semantic_scholar      :  185 records, $1.1639 total, $0.00629/record


## Step 8: Export

Save HTML report to `notebooks/results/` and extraction CSV for WU-D1 comparison.

In [15]:
# Timestamped results directory
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
results_dir = PROJECT_ROOT / "notebooks" / "results" / f"batch_abstract_evaluation_{run_id}"
results_dir.mkdir(parents=True, exist_ok=True)

# ── Extraction CSV (for WU-D1 comparison) ─────────────────────────────────
# Build flat DataFrame: one row per (record_id, source, field, true, pred, match)
detail_df = abstract_report.to_pandas()
detail_df['source'] = detail_df['record_id'].map(id_to_source)
extraction_csv = results_dir / "abstract_evaluation_detail.csv"
detail_df.to_csv(extraction_csv, index=False)
print(f"Saved detail CSV: {extraction_csv}")

# Per-field summary CSV
field_df.to_csv(results_dir / "abstract_field_metrics.csv", index=False)

# Per-source summary CSV
pd.DataFrame(source_summary).to_csv(results_dir / "abstract_source_metrics.csv", index=False)

print(f"\nAll results saved to: {results_dir}")

Saved detail CSV: /home/user/llm_metadata/notebooks/results/batch_abstract_evaluation_20260218_145142/abstract_evaluation_detail.csv

All results saved to: /home/user/llm_metadata/notebooks/results/batch_abstract_evaluation_20260218_145142


In [16]:
# HTML report via nbconvert (run after notebook is complete)
try:
    import subprocess, sys
    notebook_path = PROJECT_ROOT / "notebooks" / "batch_abstract_evaluation.ipynb"
    html_path = results_dir / "batch_abstract_evaluation.html"

    result = subprocess.run(
        [sys.executable, "-m", "nbconvert",
         "--to", "html",
         "--output", str(html_path),
         str(notebook_path)],
        capture_output=True, text=True
    )

    if result.returncode == 0:
        print(f"HTML report: {html_path}")
    else:
        print(f"nbconvert output: {result.stderr[:200]}")
        print(f"(Run nbconvert manually if needed)")
except Exception as e:
    print(f"Could not auto-generate HTML: {e}")
    print("Run manually: jupyter nbconvert --to html notebooks/batch_abstract_evaluation.ipynb")

HTML report: /home/user/llm_metadata/notebooks/results/batch_abstract_evaluation_20260218_145142/batch_abstract_evaluation.html


## Summary

**288 records evaluated** (36 Dryad + 67 Zenodo + 185 Semantic Scholar)

| Metric | Value |
|--------|-------|
| Records evaluated | 288 |
| Micro F1 (all 14 fields) | 0.202 |
| Macro F1 (all 14 fields) | 0.194 |
| Core fields Micro F1 | 0.209 |
| Modulator Micro F1 | 0.175 |
| Total cost (USD) | $1.91 |
| Avg cost / record | $0.0066 |

**Per-field highlights:**

| Field | F1 | Notes |
|-------|----|-------|
| `temp_range_i` | 0.41 | Best core field |
| `temp_range_f` | 0.38 | |
| `species` | 0.26 | High recall (0.60), low precision (0.16) — over-extraction |
| `data_type` | 0.19 | |
| `multispecies` | 0.33 | Best modulator |
| `time_series` | 0.20 | Very high recall (0.92) — model extracts too broadly |
| `referred_dataset` | 0.00 | Not in abstract text; expected |
| `bias_north_south` | 0.00 | No positive GT examples in abstract-only corpus |

**Cross-source performance:**

| Source | n | Micro F1 | Macro F1 |
|--------|---|----------|----------|
| Dryad | 36 | 0.259 | 0.237 |
| Zenodo | 67 | 0.273 | 0.329 |
| Semantic Scholar | 185 | 0.148 | 0.155 |

SS lower because abstracts are shorter/less structured (journal articles vs repository abstracts).